# Session 02: Collections Deep Dive

**Notebook 3 of 3: Collections** · used in the second half of the live session, and for revision (about 50 minutes on your own)

Python's four built-in collections (list, tuple, dict and set) do the job of most of the collection classes you use in Java, C# or TypeScript. They're built into the syntax, so idiomatic Python leans on them heavily. Choosing the right one makes code shorter, faster and more obviously correct.

By the end of this notebook you will be able to:

1. **Choose** between list, tuple, dict and set for a given access pattern, and **justify** the choice in terms of lookups, uniqueness and mutability.
2. **Sort** records by one or more fields with `key=`, and **explain** the difference between `sort()` and `sorted()`.
3. **Unpack** tuples, and **use** them as composite dict keys.
4. **Read** nested dicts safely, and **count** and **group** records with `Counter` and `defaultdict`.
5. **Implement** a multi-step transformation over nested business records: normalise, deduplicate, filter, group and aggregate.
6. **Debug** a broken report function using Observe → Hypothesize → Inspect → Fix → Verify.

> **Prerequisite:** `01-concepts.ipynb`, especially Parts 1 and 2 on names, objects and mutability.

## The engineering problem

Every night Orbital Logistics exports open and recently closed incident tickets. The on-call lead wants a morning summary:

- How many tickets are open per team, and how many have breached their SLA?
- Which tickets appear twice in the export (it happens when a ticket is updated mid-export)?
- Which tags come up most often this week?

In Java you'd reach for `ArrayList`, `HashMap`, `HashSet`, maybe a stream with `groupingBy`. Python has direct equivalents, built into the language. Here is the data we'll work with. It's synthetic and deliberately a little messy, as real exports are.

In [ ]:
TICKETS = [
    {"id": "INC-1001", "team": "platform", "priority": "P1", "status": "open",
     "age_hours": 6, "assignee": "amira", "tags": ["network", "dns"]},
    {"id": "INC-1002", "team": "database", "priority": "P3", "status": "open",
     "age_hours": 80, "assignee": None, "tags": ["backup"]},
    {"id": "INC-1003", "team": "platform", "priority": "P2", "status": "closed",
     "age_hours": 30, "assignee": "jonas", "tags": ["deploy"]},
    {"id": "INC-1004", "team": "network", "priority": "P1", "status": "open",
     "age_hours": 2, "assignee": "li", "tags": ["network", "vpn"]},
    {"id": "INC-1005", "team": "database", "priority": "P2", "status": "open",
     "age_hours": 26, "assignee": "sofia", "tags": ["latency", "network"]},
    {"id": "INC-1003", "team": "platform", "priority": "P2", "status": "closed",
     "age_hours": 31, "assignee": "jonas", "tags": ["deploy"]},
    {"id": "INC-1006", "team": "platform", "priority": "P4", "status": "open",
     "age_hours": 200, "assignee": None, "tags": []},
]

SLA_HOURS = {"P1": 4, "P2": 24, "P3": 72, "P4": 168}

print(len(TICKETS), "records loaded")

## Part 1: Choosing a collection

Before learning each type's API, learn the question that picks between them: **how will the code access this data?**

![Choosing a collection](../../../assets/diagrams/collection-selection-guide.svg)

**How this works**

- Start from the access pattern, not the data. The same ticket IDs might belong in a list (to keep export order), in a set (to check "seen it already?") or as dict keys (to find a ticket by ID).
- "Look values up by a key" wins first because it's the most common need in data code, and a dict answers it in roughly constant time however large it grows.
- Sets and dict keys use **hashing**, which is why they're fast, why they need hashable (immutable) members, and why a set has no positions to index into.
- Tuples are for **fixed-shape** groups where position has meaning: `(team, priority)`. Because they're immutable, they're safe to share and can be dict keys.
- The list is the default "ordered bag of things", and its weakness is search: `x in some_list` checks every element in turn. The warning bar is one of the most common performance bugs in Python data code, and the timing cell below shows its cost.

| Python | Java | C# | TypeScript | Notes |
|---|---|---|---|---|
| `list` | `ArrayList<T>` | `List<T>` | `T[]` | Holds any mix of types; no generics needed at run time |
| `tuple` | a `record` | `ValueTuple` / `record` | `[A, B]` tuple type | Immutable; `namedtuple` and dataclasses (Session 04) add field names |
| `dict` | `LinkedHashMap<K, V>` | `Dictionary<K, V>` | `Map<K, V>` | **Keeps insertion order** (guaranteed since Python 3.7) |
| `set` | `HashSet<T>` | `HashSet<T>` | `Set<T>` | Unordered; `frozenset` is the immutable version |

**Predict:** checking whether an ID is in a collection of 100,000 IDs, 1,000 times. How much slower will the list be than the set?

In [ ]:
import timeit

ticket_ids_list = [f"INC-{n}" for n in range(100_000)]
ticket_ids_set = set(ticket_ids_list)
lookup = "INC-99999"   # near the end: the worst case for a list

list_seconds = timeit.timeit(lambda: lookup in ticket_ids_list, number=1_000)
set_seconds = timeit.timeit(lambda: lookup in ticket_ids_set, number=1_000)

print(f"list: {list_seconds * 1000:8.2f} ms")
print(f"set:  {set_seconds * 1000:8.2f} ms")
print(f"set is roughly {list_seconds / set_seconds:,.0f}x faster here")

**Explain:** your exact numbers depend on your machine, but the set should win by a factor in the thousands. The list compares the lookup against every element until it finds a match. The set computes the string's hash and jumps straight to the right slot. With 100 items you'd never notice; with 100,000 inside a loop, it's the difference between milliseconds and minutes.

(`100_000` is just `100000`: underscores in numbers are allowed for readability. `lambda` creates a small inline function; Session 03 covers functions properly.)

## Part 2: Lists

Lists are ordered and mutable, and they're the type you'll use most. The essential operations:

In [ ]:
queue = ["INC-1001", "INC-1004"]

queue.append("INC-1005")           # add one item at the end
queue.extend(["INC-1006", "INC-1002"])  # add several
queue.insert(0, "INC-0999")        # insert at a position (slow for long lists)
last = queue.pop()                 # remove and return the last item
queue.remove("INC-1004")           # remove the first matching value (ValueError if absent)

print(queue, "| popped:", last)
print("INC-1005" in queue, queue.index("INC-1005"), len(queue))

### Sorting: `sort()` versus `sorted()`

This is the naming convention from `01-concepts.ipynb` in action:

- `items.sort()` sorts the list **in place** and returns `None`.
- `sorted(items)` returns a **new** sorted list and leaves the original alone. It accepts any iterable, not just lists.

**Predict:** what does `result` contain?

In [ ]:
priorities = ["P3", "P1", "P2"]
result = priorities.sort()
print("result:", result)
print("priorities:", priorities)

**Explain:** `result` is `None`, because `sort()` changed the list and returned nothing. Writing `items = items.sort()` is a classic bug: it replaces your list with `None`.

To sort **records**, pass a `key` function that extracts what to sort by. A tuple key sorts by several fields: by the first, then the second for ties, and so on. Tuples compare element by element, just like the version tuples in Session 01.

In [ ]:
open_tickets = [t for t in TICKETS if t["status"] == "open"]

by_age = sorted(open_tickets, key=lambda t: t["age_hours"], reverse=True)
print([(t["id"], t["age_hours"]) for t in by_age])

by_priority_then_age = sorted(open_tickets, key=lambda t: (t["priority"], -t["age_hours"]))
print([(t["id"], t["priority"], t["age_hours"]) for t in by_priority_then_age])

**Explain:** the second sort orders by priority (`"P1"` before `"P2"`, because strings compare character by character), then by age, **descending**, since negating a number reverses its order. Python's sort is also *stable*: items with equal keys keep their original relative order, so you can sort in several passes when a negation trick isn't available (for strings, say).

| Java | C# | Python |
|---|---|---|
| `list.sort(Comparator.comparing(T::getAge).reversed())` | `list.OrderByDescending(t => t.Age)` | `sorted(items, key=lambda t: t["age_hours"], reverse=True)` |

## Part 3: Tuples and unpacking

A tuple is an immutable sequence. Parentheses are optional; it's the **comma** that makes a tuple. Tuples shine in **unpacking**, which assigns several names in one statement:

In [ ]:
team, priority = ("platform", "P1")      # unpacking
print(team, priority)

first, *middle, last = ["INC-1", "INC-2", "INC-3", "INC-4"]   # star collects the rest into a list
print(first, middle, last)

primary, backup = "amira", "jonas"
primary, backup = backup, primary        # swap, no temporary variable
print(primary, backup)

single = ("P1",)                         # a one-item tuple needs the trailing comma
not_a_tuple = ("P1")
print(type(single).__name__, type(not_a_tuple).__name__)

Unpacking is also how functions return several values: `return count, oldest` returns one tuple, and the caller writes `count, oldest = summarise(...)`. And it's what `for key, value in d.items()` and `for i, item in enumerate(xs)` were doing all along.

**Tuples as composite keys.** Counting breaches per `(team, priority)` pair needs a key made of two values. In Java you'd build a string like `team + "|" + priority` or write a key class; in Python a tuple is the key:

In [ ]:
breaches = {}
for ticket in TICKETS:
    if ticket["status"] == "open" and ticket["age_hours"] > SLA_HOURS[ticket["priority"]]:
        key = (ticket["team"], ticket["priority"])
        breaches[key] = breaches.get(key, 0) + 1

print(breaches)

## Part 4: Dicts

Dicts are the workhorse of Python data code; JSON objects become dicts, HTTP headers are dict-like, and so are configuration and keyword arguments. Insertion order is preserved.

### Reading keys: `[]` versus `.get()`

`d[key]` raises `KeyError` when the key is missing. `d.get(key)` returns `None` (or a default you supply). Choose based on whether a missing key is an **error** or a **normal case**:

In [ ]:
ticket = {"id": "INC-1002", "team": "database", "assignee": None}

print(ticket.get("resolution"))                  # missing -> None
print(ticket.get("resolution", "unresolved"))    # missing -> default
print("resolution" in ticket)                    # membership tests the KEYS

try:
    ticket["resolution"]
except KeyError as error:
    print("KeyError:", error)

**Engineering Insight:** `.get()` everywhere is not "defensive", it's hiding bugs. If every ticket *must* have an `id`, write `ticket["id"]` so that a bad record fails loudly, close to the cause. Use `.get()` for fields that are genuinely optional.

Note the subtle case above: `ticket.get("assignee")` and `ticket.get("missing_key")` both return `None`. If you need to tell "present but null" from "absent", use `"assignee" in ticket`.

### Iterating and building

`for key in d`, `d.keys()`, `d.values()` and `d.items()` cover iteration. To combine dicts, `|` creates a merged copy (the right side wins on conflicts), and `|=` updates in place:

In [ ]:
default_sla = {"P1": 4, "P2": 24, "P3": 72, "P4": 168}
platform_overrides = {"P1": 2}

platform_sla = default_sla | platform_overrides
print(platform_sla)
print(default_sla["P1"], "<- unchanged")

### Reading nested data safely

Real records nest. Chained `.get()` calls work, provided each level returns a dict:

In [ ]:
event = {"ticket": {"id": "INC-1001", "sla": {"target_hours": 4}}}

print(event["ticket"]["sla"]["target_hours"])
print(event.get("ticket", {}).get("escalation", {}).get("level", "none"))

Past two levels, chained `.get()` gets hard to read, and it silently turns a misspelled key into a default. Session 06 replaces this with Pydantic models that validate the whole structure in one step. For now, prefer `[]` for required fields and `.get()` only where a level is genuinely optional.

### Counting and grouping: `Counter` and `defaultdict`

Two tools from the standard-library `collections` module remove most hand-written bookkeeping:

- `Counter` counts hashable things.
- `defaultdict(factory)` is a dict that creates a missing key's value by calling `factory()` the first time you touch it. `defaultdict(list)` is the standard group-by.

In [ ]:
from collections import Counter, defaultdict

tag_counts = Counter(tag for ticket in TICKETS for tag in ticket["tags"])
print(tag_counts.most_common(3))

ids_by_team = defaultdict(list)
for ticket in TICKETS:
    ids_by_team[ticket["team"]].append(ticket["id"])
print(dict(ids_by_team))

**Explain:** the `Counter` line uses a generator expression with **two** `for` clauses, read left to right like nested loops: for each ticket, for each tag in that ticket. That's the limit for comprehension nesting; any deeper and a loop reads better.

Without `defaultdict`, the grouping loop needs an `if team not in ids_by_team: ids_by_team[team] = []` check on every iteration. `dict.setdefault(key, []).append(...)` is the plain-dict alternative you'll see in older code.

| Task | Java | C# | Python |
|---|---|---|---|
| Group by | `Collectors.groupingBy(...)` | `.GroupBy(...)` | `defaultdict(list)` and a loop |
| Count by | `groupingBy(..., counting())` | `.GroupBy(...).ToDictionary(g => g.Key, g => g.Count())` | `Counter(...)` |

## Part 5: Sets

Sets hold unique, hashable items with no order. Use them for **membership**, **deduplication** and **comparing groups**:

In [ ]:
yesterday = {"INC-1001", "INC-1002", "INC-1004"}
today = {"INC-1002", "INC-1004", "INC-1005", "INC-1006"}

print("new today:     ", sorted(today - yesterday))   # difference
print("resolved:      ", sorted(yesterday - today))
print("still open:    ", sorted(today & yesterday))   # intersection
print("seen at all:   ", sorted(today | yesterday))   # union
print("in only one:    ", sorted(today ^ yesterday))   # symmetric difference

Results are wrapped in `sorted()` only so that the output is stable to read. Sets have no defined order, and printing one directly can show a different order from run to run.

**Common Mistake:** `{}` is an empty **dict**, not an empty set. Write `set()`.

**Deduplicating while keeping order.** `set(items)` removes duplicates but loses the order. When order matters (as in an export), `dict.fromkeys()` keeps the first occurrence of each, because dict keys are unique **and** ordered:

In [ ]:
export_order = ["INC-1003", "INC-1001", "INC-1003", "INC-1002", "INC-1001"]

print(list(dict.fromkeys(export_order)))

For records, rather than plain values, you need to dedupe **by a field**. A set of seen IDs is the standard pattern:

In [ ]:
seen_ids = set()
unique_tickets = []
for ticket in TICKETS:
    if ticket["id"] not in seen_ids:      # O(1) check, thanks to the set
        seen_ids.add(ticket["id"])
        unique_tickets.append(ticket)

print(len(TICKETS), "->", len(unique_tickets), "tickets after removing duplicates")

## Part 6: Putting it together

Now the morning summary from the engineering problem. A pipeline of small, separate steps is easier to read and test than one big loop, and each step produces **new** data rather than editing the old:

1. **Deduplicate** by ticket ID (keep the first).
2. **Filter** to open tickets.
3. **Group** by team.
4. **Aggregate** per team: open count, SLA breaches, oldest ticket.
5. **Rank** teams by breaches.

In [ ]:
from collections import defaultdict

# 1. Deduplicate (as in Part 5)
seen_ids = set()
unique_tickets = []
for ticket in TICKETS:
    if ticket["id"] not in seen_ids:
        seen_ids.add(ticket["id"])
        unique_tickets.append(ticket)

# 2. Filter
open_tickets = [t for t in unique_tickets if t["status"] == "open"]

# 3. Group
by_team = defaultdict(list)
for ticket in open_tickets:
    by_team[ticket["team"]].append(ticket)

# 4. Aggregate
summary = {
    team: {
        "open": len(team_tickets),
        "breached": sum(t["age_hours"] > SLA_HOURS[t["priority"]] for t in team_tickets),
        "oldest_hours": max(t["age_hours"] for t in team_tickets),
    }
    for team, team_tickets in by_team.items()
}

# 5. Rank: most breaches first, then team name for a stable order
for team, stats in sorted(summary.items(), key=lambda item: (-item[1]["breached"], item[0])):
    print(
        f"{team:<10} open={stats['open']}  breached={stats['breached']}"
        f"  oldest={stats['oldest_hours']}h"
    )

**Explain** a few details:

- `sum(... > ... for t in ...)` counts the `True` values, because `True` is `1` (Part 3 of `01-concepts.ipynb`).
- `max()` on an empty sequence raises `ValueError`. It's safe here because a team only appears in `by_team` if it has at least one ticket. If you compute it elsewhere, use `max(..., default=None)`.
- `TICKETS` is unchanged after all this. Every step built new lists and dicts; nothing was edited in place.

**Modify:** add an `"unassigned"` count to each team's stats (an assignee of `None` or `""` counts as unassigned). Which test from `01-concepts.ipynb` Part 4 is right here: truthiness, or `is None`?

In [ ]:
# Your turn: copy the summary comprehension here and add an "unassigned" count.

## Part 7: Debugging lab, a report that lies

A colleague wrote this helper for the dashboard. It has **three** bugs. Run it and work through them using **Observe → Hypothesize → Inspect → Fix → Verify**. Resist reading the hints until you've formed your own hypothesis.

In [ ]:
import copy

source_tickets = copy.deepcopy(TICKETS)   # a private copy, so this lab can't damage TICKETS


def oldest_open_per_team(tickets):
    """Return {team: [ids of open tickets, oldest first]} without changing `tickets`."""
    teams = {t["team"] for t in tickets}
    result = dict.fromkeys(teams, [])
    tickets.sort(key=lambda t: t["age_hours"], reverse=True)
    for ticket in tickets:
        if ticket["status"] == "open":
            result[ticket["team"]].append(ticket["id"])
    return result


before = [t["id"] for t in source_tickets]
report = oldest_open_per_team(source_tickets)
after = [t["id"] for t in source_tickets]

for team in sorted(report):
    print(f"{team:<10} {report[team]}")
print("Input order unchanged?", before == after)

**Observe:** every team shows the *same* list, containing tickets from all teams. And the input order changed, although the docstring promises it won't.

<details>
<summary>Hint 1: why do all teams share one list?</summary>

`dict.fromkeys(teams, [])` evaluates `[]` **once**, and then uses that one list object as the value for every key. It's aliasing again (`01-concepts.ipynb`, Part 1). **Inspect:** print `result["platform"] is result["network"]` right after creating it.

**Fix:** create a separate list per key: `{team: [] for team in teams}`, or use `defaultdict(list)`.

</details>

<details>
<summary>Hint 2: why did the input change?</summary>

`tickets.sort(...)` sorts the caller's list in place: the function was handed a reference to it, not a copy. **Fix:** loop over `sorted(tickets, key=..., reverse=True)`, which leaves the input alone.

</details>

<details>
<summary>Hint 3: the third bug only shows up with different data</summary>

Duplicates. `INC-1003` appears twice in the export. It's closed, so it's filtered out today, but an open duplicate would be listed twice. **Fix:** deduplicate first, or skip IDs already seen with a set. **Verify** by adding an open duplicate to `source_tickets` and re-running.

</details>

Write your fixed version in the next cell, and **verify** that each team has its own list, oldest first, with no duplicates, and that `before == after`.

In [ ]:
# Your fixed version of oldest_open_per_team goes here.

## Common mistakes

| Mistake | What happens | Instead |
|---|---|---|
| `x in big_list` inside a loop | Every check scans the list | Build a `set` (or dict) once, before the loop |
| `items = items.sort()` | `items` becomes `None` | `items.sort()` alone, or `items = sorted(items)` |
| Sorting the caller's list inside a helper | The caller's data is reordered | `sorted(...)` returns a new list |
| `dict.fromkeys(keys, [])` | Every key shares **one** list | `{k: [] for k in keys}` or `defaultdict(list)` |
| `d.get(...)` for required fields | Bad records slip through as `None` | `d[...]` for required fields, so failures are loud |
| `{}` for an empty set | It's an empty dict | `set()` |
| Depending on set order | Order can differ between runs | `sorted(the_set)` when order matters |
| `set(items)` to dedupe an ordered export | Original order lost | `list(dict.fromkeys(items))`, or a seen-set loop |
| `max(...)` on a possibly empty sequence | `ValueError` | `max(..., default=None)` |
| `("P1")` as a one-item tuple | It's just the string `"P1"` | `("P1",)` |

## Key takeaways

- Pick a collection by **access pattern**: key lookup → dict; membership or uniqueness → set; fixed-shape record or composite key → tuple; otherwise → list.
- `x in list` is a scan; `x in set` and `key in dict` are hash lookups. The difference grows with the data.
- `sorted()` returns a new list; `.sort()` sorts in place and returns `None`. Sort records with `key=`, and by several fields with a tuple key.
- Unpacking (`a, b = pair`, `first, *rest = items`) is everywhere: loops, function returns, swaps.
- Use `d[key]` for required fields and `d.get(key, default)` for optional ones.
- `Counter` counts, `defaultdict(list)` groups, and set operators (`-`, `&`, `|`) compare groups.
- Build transformations as small steps that return **new** data. Your inputs stay intact, and each step can be tested on its own.

## Further reading

See [`../resources/references.md`](../resources/references.md), especially the tutorial chapter on data structures and the `collections` module documentation.

**Next:** test yourself with **`02-practice-quiz.ipynb`**, then start the workbook assignment in [`../workbook/assignment.md`](../workbook/assignment.md).